# RFFMG: Representation for Flexible Fragment-Controlled Molecular Generation

本ノートブックでは、RFFMGの使い方をステップごとに解説します。

## 全体の流れ

1. **環境構築** — conda環境のセットアップ
2. **データのキュレーション** — ChEMBL化合物のフィルタリング・標準化
3. **フラグメンテーション** — 分子をフラグメントに分解
4. **SAFE表現への変換** — SAFE形式のフラグメントデータ作成
5. **データセット分割** — train/val/test splitの作成
6. **モデルの学習** — T5Chem / SAFE-GPTの学習
7. **分子生成** — 学習済みモデルによる分子生成
8. **評価** — 生成分子の品質評価

---
## 0. 環境構築

本プロジェクトでは **2つのconda環境** が必要です。  
T5ChemとSAFEは依存関係が異なるため、別々の環境で動作させます。

### リポジトリのクローン
```bash
git clone https://github.com/sa-akinori/mol_gen_frags.git
cd mol_gen_frags
```

### T5Chem環境
```bash
conda create -n t5chem_copy python=3.12
conda activate t5chem_copy
pip install -r requirements/t5chem_requirements.txt
pip install -e .
```

### SAFE環境
```bash
conda create -n safe_copy python=3.12
conda activate safe_copy
pip install -r requirements/safe_requirements.txt
pip install -e .
```

> **注意**: 以降のセルを実行する際は、各ステップで指定された環境が有効であることを確認してください。  
> Jupyter上でカーネルを切り替えるか、ターミナルから該当環境で実行してください。

---
## 1. 事前学習モデル・データの準備

学習済みモデルをHugging Faceからダウンロードするか、事前学習モデルを個別に取得します。

### 1-a. 本研究の学習済みモデル（Hugging Face）

すでに学習済みのモデルを使いたい場合、以下でダウンロードできます。

In [ ]:
# Hugging Faceから学習済みモデルをダウンロード
!python -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='sato-akinori/FFMG', allow_patterns='models/*', local_dir='.')"

# zipファイルを解凍
!shopt -s globstar; for zip in models/**/*.zip; do unzip -o "$zip" -d "$(dirname "$zip")"; done
!find models -name "*.zip" -exec sh -c 'unzip -o "$1" -d "$(dirname "$1")" && rm "$1"' _ {} \;

### 1-b. 事前学習モデルの個別ダウンロード

自分でファインチューニングする場合に必要な事前学習モデルです。

In [ ]:
# T5Chem 事前学習モデル (Zenodo)
!mkdir -p models/t5chem/pretrained
!wget -P models/t5chem/pretrained https://zenodo.org/records/14280768/files/simple_pretrain.tar.bz2
!tar -xjvf models/t5chem/pretrained/simple_pretrain.tar.bz2 --strip-components=3 -C models/t5chem/pretrained/

In [ ]:
# SAFE-GPT 事前学習モデル (Hugging Face)
!mkdir -p models/safe_gpt/pretrained
!git clone https://huggingface.co/datamol-io/safe-gpt/ models/safe_gpt/pretrained/

---
## 2. データのキュレーション

ChEMBL31データセットから化合物をフィルタリングし、標準化します。  
以下の処理が行われます:

- RDKitによるcanonical SMILES変換
- 重原子数によるサイズフィルタ（95パーセンタイル / 下限5）
- 構造アラート除去（Glaxo, PAINSフィルタ）
- 分子量 > 1000 の除外

> **使用環境**: `t5chem_copy`

In [ ]:
# curate_datasets.py の実行
# 事前にChEMBL31データ (data/all_curated_cpds_chembl31.tsv) を配置してください
!python src/curate_datasets.py

キュレーション結果は `data/curated/` に保存されます:
- `passed_filters_rdkit_canonical_smiles.tsv` — フィルタを通過した化合物
- `compound_curation.txt` — 各ステップのログ

---
## 3. フラグメンテーション（RFFMGフラグメントの作成）

キュレーション済み分子をフラグメントに分解します。  
フラグメンテーション手法は **2種類** から選択できます:

| 手法 | 説明 |
|------|------|
| `rc_cms` | SP3-SP3炭素間結合やリング間結合をランダムに切断 |
| `brics` | BRICS (Breaking of Retrosynthetically Interesting Chemical Substructures) による切断 |

> **使用環境**: `t5chem_copy`

In [ ]:
# フラグメンテーション手法を選択
FRAG_METHOD = "rc_cms"  # "brics" または "rc_cms"

In [ ]:
!python src/gen_frags/rffmg_frags.py --frag_method {FRAG_METHOD}

### フラグメンテーションの仕組み（詳細）

`Smi2SentenceOpt` で制御される主要パラメータ:

| パラメータ | デフォルト値 | 説明 |
|-----------|------------|------|
| `fragmentRatio` | 0.6 | 切断対象結合のうち、実際に切断する割合 |
| `bigRingThres` | 7 | このサイズ以上のリング内結合を切断対象にする |
| `nFragmentPatterns` | 5 | 1分子あたりのフラグメンテーションパターン数 |
| `nSamplingTrialsPerFragset` | 5 | フラグメントセットあたりのサンプリング試行回数 |
| `uppMolSizeToFragSize` | 1.75 | フラグメント/分子のサイズ比上限 |

### フラグメンテーションの動作例

1分子に対してフラグメンテーションがどのように動作するか確認できます。

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from func.fragmentation import RandomFragmentize, BRICSFragmentize, PostProcessSelectFrags, Smi2Sentences, Smi2SentenceOpt

# 例: アスピリン
smi = "CC(=O)Oc1ccccc1C(=O)O"
mol = Chem.MolFromSmiles(smi)

print(f"入力分子: {smi}")
display(Draw.MolToImage(mol, size=(300, 200)))

In [ ]:
# rc_cms によるフラグメンテーション
frags_rc = RandomFragmentize(mol, returnSmiles=False, bigRingThres=7, ratio=0.6, rseed=42)
if frags_rc is not None:
    passfrags, _ = PostProcessSelectFrags(frags_rc, returnAsSmi=True)
    print(f"rc_cms フラグメント: {passfrags}")
    frag_mols = [Chem.MolFromSmiles(f) for f in passfrags.split('.')]
    display(Draw.MolsToGridImage(frag_mols, molsPerRow=4, subImgSize=(250, 200)))
else:
    print("rc_cmsで切断可能な結合が見つかりませんでした")

In [ ]:
# BRICS によるフラグメンテーション
frags_brics = BRICSFragmentize(mol, returnSmiles=False)
if frags_brics is not None:
    passfrags_b, _ = PostProcessSelectFrags(frags_brics, trimRgroupOnRing=False, returnAsSmi=True)
    print(f"BRICS フラグメント: {passfrags_b}")
    frag_mols_b = [Chem.MolFromSmiles(f) for f in passfrags_b.split('.')]
    display(Draw.MolsToGridImage(frag_mols_b, molsPerRow=4, subImgSize=(250, 200)))
else:
    print("BRICSで切断可能な結合が見つかりませんでした")

In [ ]:
# Smi2Sentences を使って fragment >> molecule のペアを生成
opt = Smi2SentenceOpt(
    fragmentMethod='rc_cms',
    fragmentRatio=0.6,
    bigRingThres=7,
    nFragmentPatterns=5,
    nSamplingTrialsPerFragset=5,
    uppMolSizeToFragSize=1.75,
    uniqunize=False
)
s2s = Smi2Sentences(opt)
results, _, _ = s2s(smi)

print(f"生成されたフラグメント→分子ペア数: {len(results)}")
for i, pair in enumerate(results[:5]):
    frag, mol_smi = pair.split('>>')
    print(f"  [{i+1}] {frag} >> {mol_smi}")

---
## 4. SAFE表現への変換

RFFMGフラグメントをSAFE (Sequential Attachment-based Fragment Embedding) 形式に変換します。  
SAFE-GPTモデルの学習に使用します。

> **使用環境**: `safe_copy`

In [ ]:
!python src/gen_frags/safe_frags.py --frag_method {FRAG_METHOD}

変換結果は `data/safe/{FRAG_METHOD}/safe_smiles.csv` に保存されます。  
各行にはSMILES、SAFE表現（full/pass）、フラグメント情報が含まれます。

---
## 5. データセット分割（Train / Validation / Test）

RFFMG（T5Chem用）とSAFE（SAFE-GPT用）の両方のデータセットを作成します。  
分割比率: **train 95% / validation 2.5% / test 2.5%**

同時に、以下の評価用データセットも作成されます:
- `frag_order` — フラグメント順序のロバスト性評価用
- `frag_num` — フラグメント数のロバスト性評価用
- `dup_frags` — 重複フラグメントのロバスト性評価用
- `attach_point_num` — アタッチメントポイント数の評価用

> **使用環境**: `safe_copy`

In [ ]:
!python src/make_datasets.py --frag_method {FRAG_METHOD}

### 出力ファイル構成

```
data/
├── rffmg/{FRAG_METHOD}/
│   ├── full_dataset.csv          # 全フラグメントデータ
│   ├── unique_frags.csv          # ユニークフラグメント一覧
│   ├── normal/                   # 通常のtrain/val/test split
│   │   ├── train.source / train.target
│   │   ├── val.source / val.target
│   │   └── test.source / test.target
│   ├── frag_order/               # フラグメント順序評価用
│   ├── frag_num/                 # フラグメント数評価用
│   ├── dup_frags/                # 重複フラグメント評価用
│   └── attach_point_num/         # アタッチメントポイント評価用
│
└── safe/{FRAG_METHOD}/
    ├── safe_smiles.csv           # SAFE変換済みデータ
    └── normal/                   # HuggingFace Dataset形式
```

---
## 6. モデルの学習

2つのアーキテクチャで学習を行います。

### 6-a. T5Chem（RFFMG）の学習

T5ベースのTransformerモデルを用いて、フラグメント→分子の変換を学習します。

| パラメータ | 値 | 説明 |
|-----------|-----|------|
| `--data_dir` | `data/rffmg/{method}/normal` | 学習データのディレクトリ |
| `--pretrain` | `models/t5chem/pretrained` | 事前学習モデルのパス（`''`で事前学習なし） |
| `--task_type` | `product` | タスクの種類 |
| `--num_epoch` | `50` | エポック数 |

> **使用環境**: `t5chem_copy`

In [ ]:
# T5Chemの学習
!t5chem train \
    --data_dir data/rffmg/{FRAG_METHOD}/normal \
    --output_dir models/t5chem/trained/rffmg/{FRAG_METHOD} \
    --pretrain models/t5chem/pretrained \
    --task_type product \
    --num_epoch 50

### 6-b. SAFE-GPTのファインチューニング

SAFE-GPTモデルをフラグメントデータでファインチューニングします。  
引数が多いため、シェルスクリプト (`src/train_model/run_safe.sh`) で実行します。

主要パラメータ:

| パラメータ | 値 | 説明 |
|-----------|-----|------|
| `--config` | `models/safe_gpt/pretrained/config.json` | モデル設定 |
| `--tokenizer` | `models/safe_gpt/pretrained/tokenizer.json` | トークナイザー |
| `--text_column` | `full_safe` | 学習対象のカラム |
| `--num_train_epochs` | `50` | エポック数 |
| `--learning_rate` | `1e-4` | 学習率 |

> **使用環境**: `safe_copy`

In [ ]:
# SAFE-GPTのファインチューニング
# FRAG_METHODに合わせてrun_safe.sh内のFRAG_NAMEを変更してください
!bash src/train_model/run_safe.sh

> **Note**: 事前学習なしで学習する場合（from_scratch）は、`--pretrain ''` を指定してください。

---
## 7. 分子生成

学習済みモデルを使って、フラグメントから分子を生成します。  
どちらのモデルも **ビームサーチ** を使用し、1入力あたり50個の分子を生成します。

### 7-a. T5Chemモデルによる分子生成

> **使用環境**: `t5chem_copy`

In [ ]:
# T5Chemモデルによる分子生成
MODEL_VER = "trained"       # "trained" または "pretrained"
ADDITIONAL_PATH = "normal"  # "normal", "dup_frags", "frag_num", "frag_order", "attach_point_num"

!python src/gen_mols/gen_t5chem.py \
    --frag_method {FRAG_METHOD} \
    --model_ver {MODEL_VER} \
    --additional_path {ADDITIONAL_PATH} \
    --n_samples 50 \
    --num_beams 50 \
    --batch_size 24 \
    --random_seed 42

### 7-b. SAFE-GPTモデルによる分子生成

> **使用環境**: `safe_copy`

In [ ]:
# SAFE-GPTモデルによる分子生成
MODEL_VER = "trained"  # "trained" または "pretrained"

!python src/gen_mols/gen_safe.py \
    --frag_method {FRAG_METHOD} \
    --model_ver {MODEL_VER} \
    --n_samples 50 \
    --num_beams 50 \
    --random_seed 42

---
## 8. 生成分子の評価

生成された分子の品質を以下の指標で評価します:

| 指標 | 説明 |
|------|------|
| **Validity** | 生成SMILESのうち有効な分子の割合 |
| **Validity on fragments** | 入力フラグメントを全て含む分子の割合 |
| **Uniqueness** | 有効な分子のうちユニークな分子の割合 |
| **Novelty** | ユニーク分子のうち学習データに含まれない分子の割合 |
| **Tanimoto similarity** | 生成分子間の平均Tanimoto類似度 |
| **SA Score** | 合成容易性スコア |

> **使用環境**: `safe_copy`

In [ ]:
# T5Chemモデルの評価
!python src/evaluation.py \
    --model_name t5chem \
    --model_ver trained \
    --frag_method {FRAG_METHOD} \
    --additional_path normal

In [ ]:
# SAFE-GPTモデルの評価
!python src/evaluation.py \
    --model_name safe_gpt \
    --model_ver trained \
    --frag_method {FRAG_METHOD} \
    --additional_path normal

### 評価結果の確認

In [ ]:
import pandas as pd

# T5Chem の統計情報
t5_stats_path = f"results/t5chem/trained/rffmg/{FRAG_METHOD}/beam/normal/stats.csv"
try:
    t5_stats = pd.read_csv(t5_stats_path, index_col=0, header=None)
    t5_stats.columns = ['T5Chem']
    print("=== T5Chem 評価結果 ===")
    display(t5_stats)
except FileNotFoundError:
    print(f"{t5_stats_path} が見つかりません。先に評価を実行してください。")

In [ ]:
# SAFE-GPT の統計情報
safe_stats_path = f"results/safe_gpt/trained/safe/{FRAG_METHOD}/beam/normal/stats.csv"
try:
    safe_stats = pd.read_csv(safe_stats_path, index_col=0, header=None)
    safe_stats.columns = ['SAFE-GPT']
    print("=== SAFE-GPT 評価結果 ===")
    display(safe_stats)
except FileNotFoundError:
    print(f"{safe_stats_path} が見つかりません。先に評価を実行してください。")

---
## 補足: 仮想環境への変更点

T5ChemおよびSAFEライブラリに対して以下の修正が必要です。  
これらは学習速度の向上やモデル保存の改善を目的としています。

### T5Chem (`t5chem/run_trainer.py`)

```python
# 1. 学習速度向上: compute_metricsを無効化
# compute_metrics = AccuracyMetrics
compute_metrics = None

# 2. モデル保存先の明確化
# tokenizer.save_vocabulary(args.output_dir)
# trainer.save_model(args.output_dir)
os.makedirs(f'{args.output_dir}/best_model/')
tokenizer.save_vocabulary(f'{args.output_dir}/best_model/')
trainer.save_model(f'{args.output_dir}/best_model/')

# 3. wandb無効化 (不要な場合)
# report_to="wandb",
report_to='none',
```

### SAFE (`safe/trainer/cli.py`)

```python
# 1. モデル保存先の明確化
# trainer.save_model()
trainer.save_model(os.path.join(training_args.output_dir, "best_model"))

# tokenizer.save(os.path.join(training_args.output_dir, "tokenizer.json"))
tokenizer.save(os.path.join(training_args.output_dir, "best_model/tokenizer.json"))

# 2. EarlyStoppingの追加
callbacks=[EarlyStoppingCallback(early_stopping_patience=15)]
```

### SAFE バージョン互換性の修正

```python
# safe/trainer/trainer_utils.py (19行目)
# def compute_loss(self, model, inputs, return_outputs=False):
def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):

# safe/tokenizer.py (290行目)
# self.tokenizer.save_pretrained(*args, **kwargs)
self.tokenizer.save(*args, **kwargs)
```